# Vergelijk post-processed GeoPackages

`wocu_post_processed_fase2_20260310.gpkg` vs `wocu_post_processed_fase2_20260330.gpkg`

Laagstructuur, rijen, kolommen en CRS per laag; verschillen tussen de twee versies.

**Let op:** het notebook laadt elke laag volledig (zoals `03_compare_geopackages.ipynb`); bij grote bestanden kan dat enkele minuten duren.

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd()
for _candidate in [_cwd, *_cwd.parents]:
    if (_candidate / 'src').exists():
        _backend = _candidate
        break
else:
    raise RuntimeError('Run from backend/ or a subfolder that contains src/')
if str(_backend) not in sys.path:
    sys.path.insert(0, str(_backend))

import pandas as pd
import geopandas as gpd
from pyogrio import list_layers

import src.paths as PATHS

DATA_DIR = PATHS.DATA_DIR
GPKG_20260310 = DATA_DIR / '02_processed/erosion/wocu_post_processed_fase2_20260310.gpkg'
GPKG_20260330 = DATA_DIR / '02_processed/erosion/wocu_post_processed_fase2_20260330.gpkg'

for p in (GPKG_20260310, GPKG_20260330):
    if p.exists():
        mb = p.stat().st_size / 1e6
        print(f'OK  {p}  ({mb:.1f} MB)')
    else:
        print(f'MISSING: {p}')

: 

In [ ]:
def layer_inventory(gpkg_path: Path) -> pd.DataFrame:
    """Eén rij per laag: naam, geometrietype, aantal rijen, kolommen."""
    rows = []
    for name, geom in list_layers(gpkg_path):
        gdf = gpd.read_file(gpkg_path, layer=name)
        rows.append({
            'layer': name,
            'geom_type': str(geom) if geom is not None else 'None',
            'n_rows': len(gdf),
            'columns': sorted(c for c in gdf.columns if c != 'geometry'),
        })
    return pd.DataFrame(rows)


def layer_summary(gdf: gpd.GeoDataFrame) -> dict:
    geom_types = gdf.geometry.geom_type.value_counts().to_dict() if len(gdf) else {}
    return {
        'n_rows': len(gdf),
        'crs': str(gdf.crs) if gdf.crs is not None else None,
        'geom_types': geom_types,
        'columns': list(gdf.columns),
    }


def read_layer(path: Path, layer: str) -> gpd.GeoDataFrame:
    return gpd.read_file(path, layer=layer)

## 1 — Lagen en feature-counts (uit metadata)

In [ ]:
if not GPKG_20260310.exists() or not GPKG_20260330.exists():
    raise FileNotFoundError(
        'Eén of beide .gpkg ontbreekt onder DATA_DIR; zet de bestanden in backend/data/02_processed/erosion/'
    )

print('Laden 20260310 …')
inv_10 = layer_inventory(GPKG_20260310).assign(source='20260310')
print('Laden 20260330 …')
inv_30 = layer_inventory(GPKG_20260330).assign(source='20260330')
display(inv_10)
display(inv_30)

In [ ]:
names_10 = set(inv_10['layer'])
names_30 = set(inv_30['layer'])
print('Alleen in 20260310:', sorted(names_10 - names_30))
print('Alleen in 20260330:', sorted(names_30 - names_10))
print('Gemeenschappelijke lagen:', sorted(names_10 & names_30))

## 2 — Per gemeenschappelijke laag: rijen, CRS, kolommen

In [ ]:
common = sorted(names_10 & names_30)
rows = []
for layer in common:
    g10 = read_layer(GPKG_20260310, layer)
    g30 = read_layer(GPKG_20260330, layer)
    s10, s30 = layer_summary(g10), layer_summary(g30)
    cols_10, cols_30 = set(g10.columns), set(g30.columns)
    rows.append({
        'layer': layer,
        'n_20260310': s10['n_rows'],
        'n_20260330': s30['n_rows'],
        'delta_n': s30['n_rows'] - s10['n_rows'],
        'crs_10': s10['crs'],
        'crs_30': s30['crs'],
        'crs_match': s10['crs'] == s30['crs'],
        'cols_only_10': sorted(cols_10 - cols_30),
        'cols_only_30': sorted(cols_30 - cols_10),
        'n_common_cols': len(cols_10 & cols_30),
    })

cmp = pd.DataFrame(rows)
display(cmp)

## 3 — Dtype-vergelijking voor gemeenschappelijke kolommen (per laag)

In [ ]:
dtype_diffs = []
for layer in common:
    g10 = read_layer(GPKG_20260310, layer)
    g30 = read_layer(GPKG_20260330, layer)
    shared = [c for c in g10.columns if c in g30.columns and c != 'geometry']
    for c in shared:
        t10, t30 = g10[c].dtype, g30[c].dtype
        if t10 != t30:
            dtype_diffs.append({'layer': layer, 'column': c, 'dtype_20260310': t10, 'dtype_20260330': t30})

if dtype_diffs:
    display(pd.DataFrame(dtype_diffs))
else:
    print('Geen dtype-verschillen voor gedeelde niet-geometriekolommen.')

## 4 — Optioneel: steekproef eerste laag met join-key

Pas `JOIN_COL` aan als er een stabiele ID-kolom is (bijv. `position_id` of `location_id`).

In [ ]:
# Voorbeeld: als beide een ID-kolom hebben, uncomment en pas aan
# JOIN_COL = 'position_id'
# layer = common[0]
# g10 = read_layer(GPKG_20260310, layer)
# g30 = read_layer(GPKG_20260330, layer)
# if JOIN_COL in g10.columns and JOIN_COL in g30.columns:
#     merged = g10[[JOIN_COL]].merge(
#         g30[[JOIN_COL]], on=JOIN_COL, how='outer', indicator=True
#     )
#     print(merged['_merge'].value_counts())
# else:
#     print('Geen JOIN_COL in beide — inspecteer kolommen hierboven.')
print('Tip: bekijk cmp["cols_only_10"] / cmp["cols_only_30"] voor schemaverschillen.')